# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields/columns
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '<no name>')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print('  Fields:')
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field.get('@id', '?')} (name: {field.get('name', '<no name>')})")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by their `@id`
record_set_ids = [rs["@id"] for rs in dataset.record_sets]
print('Available record set @id s:', record_set_ids)
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()
# Show columns of the first non-empty DataFrame
chosen_rs_id = None
for rs_id in record_set_ids:
    if not dataframes[rs_id].empty:
        chosen_rs_id = rs_id
        break
if chosen_rs_id is not None:
    print(f"Data columns in record set '{chosen_rs_id}':\n", dataframes[chosen_rs_id].columns.tolist())
    display(dataframes[chosen_rs_id].head())
else:
    print('No data in any record set!')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For EDA, choose a numeric field from the loaded DataFrame
df = dataframes[chosen_rs_id]
print('Columns:', df.columns.tolist())
numeric_field = None
for col in df.columns:
    # Try selecting the first numeric column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if numeric_field is None:
    # Try to coerce a suitable column
    for col in df.columns:
        try:
            df_temp = pd.to_numeric(df[col], errors='coerce')
            if df_temp.notna().sum() > 0:
                df[col] = df_temp
                numeric_field = col
                break
        except Exception:
            continue

if numeric_field:
    threshold = df[numeric_field].mean()  # Use mean as threshold for demo
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try grouping by a non-numeric field
    group_field = None
    for col in df.columns:
        if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print('No suitable numeric field for EDA in this record set.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    if group_field in df.columns:
        # Plot mean of numeric field by group
        grouped = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10, 5))
        sns.barplot(x=grouped.index, y=grouped.values)
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.title(f'Mean {numeric_field} by {group_field} (Top 10)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('No suitable numeric field for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset metadata and structure were extracted using `mlcroissant`, referencing all key entities via their `@id` fields.
- At least one record set was loaded and explored; numeric and categorical fields were processed for basic EDA.
- Data visualizations suggest potential for more advanced statistics or bioinformatics analysis of protein features, abundance, or experimental groups, as available in the provided Croissant schema.

For deeper insights, refer to the dataset's FAIR schema documentation and field definitions via their `@id` values.